# Laboratorium 2: Ramy, skrypty i sieci powiązań

Ramy Minsky'ego ze slotami, fasetami, dziedziczeniem i demonami, skrypty Schanka oraz rozprzestrzenianie aktywacji w sieci powiązań. Wszystko w czystym Pythonie, z networkx do grafów, rdflib i owlrl do porównania z OWL oraz angielskim WordNetem z nltk.

In [ ]:
# rdflib i owlrl nie są w Colabie domyślnie; networkx, nltk, pandas i matplotlib są
%pip install -q rdflib owlrl

In [ ]:
import itertools
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import nltk
from nltk.corpus import wordnet as wn
from scipy.stats import spearmanr
from rdflib import Graph, Namespace
import owlrl
%matplotlib inline

nltk.download("wordnet", quiet=True)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 60)

RANDOM_STATE = 42   # używany tylko przy układzie grafu do rysunku
ROK_BIEZACY = 2026  # stały rok, żeby wyniki były powtarzalne

## Zadanie 1: System ram

In [ ]:
# Szkielet systemu ram. Uzupełnijcie metody local_answer i lookup.

class _Missing:
    def __repr__(self):
        return "BRAK"

MISSING = _Missing()
FACETS = {"value", "default", "type", "if_needed", "if_added", "card"}


class Frame:
    check_changes = True   # w zadaniu 3 wyłączamy, żeby zobaczyć pętlę demonów
    demon_calls = 0        # licznik wywołań demonów if-added

    def __init__(self, name, parents=(), instance=False):
        self.name = name
        self.instance = instance
        self.parents = list(parents)
        for p in self.parents:
            if p.instance:
                raise ValueError(f"{p.name} jest instancją i nie może być rodzicem")
        self.slots = {}   # slot -> {faseta: wartość}

    def __repr__(self):
        return self.name

    def define(self, slot, **facets):
        unknown = set(facets) - FACETS
        if unknown:
            raise ValueError(f"nieznane fasety: {unknown}")
        self.slots.setdefault(slot, {}).update(facets)
        return self

    def own(self, slot, facet):
        return self.slots.get(slot, {}).get(facet, MISSING)

    def is_a(self, other):
        """Czy rama jest (przechodnio) rodzajem albo instancją ramy other."""
        return self is other or any(p.is_a(other) for p in self.parents)

    def linearize(self, strategy="mro"):
        """Kolejność przeszukiwania przodków, łącznie z samą ramą."""
        if strategy == "dfs":
            order = []
            def visit(f):
                if f not in order:
                    order.append(f)
                    for p in f.parents:
                        visit(p)
            visit(self)
            return order
        if strategy == "mro":
            return self._c3()
        raise ValueError(strategy)

    def _c3(self):
        # linearyzacja C3, ta sama, której Python używa w __mro__
        seqs = [p._c3() for p in self.parents] + [list(self.parents)]
        result = [self]
        while True:
            seqs = [s for s in seqs if s]
            if not seqs:
                return result
            for s in seqs:
                head = s[0]
                if not any(head in t[1:] for t in seqs):
                    break
            else:
                raise TypeError(f"brak spójnej kolejności dziedziczenia dla {self.name}")
            result.append(head)
            for t in seqs:
                if t[0] is head:
                    del t[0]

    def inherited_facet(self, slot, facet):
        """Faseta opisująca slot (type, card, if_added) z pierwszej ramy w kolejności MRO."""
        for f in self.linearize("mro"):
            v = f.own(slot, facet)
            if v is not MISSING:
                return v
        return MISSING

    def local_answer(self, frame, slot):
        """Odpowiedź jednej ramy z łańcucha: (wartość, faseta) albo (MISSING, None).
        Kolejność: value, demon if_needed wywołany dla self (nie dla frame!), default.
        Demon zwracający None traktujemy jak brak odpowiedzi."""
        # UZUPEŁNIJ
        raise NotImplementedError

    def lookup(self, slot, strategy="mro"):
        """Zwraca (wartość, rama źródłowa, faseta). Brak wartości: (None, None, None).
        Przeglądaj ramy w kolejności self.linearize(strategy) i użyj local_answer."""
        # UZUPEŁNIJ
        raise NotImplementedError

    def get(self, slot, strategy="mro"):
        return self.lookup(slot, strategy)[0]

    def put(self, slot, value):
        """Wpisuje wartość do fasety value z kontrolą typu i liczności, potem woła if_added."""
        typ = self.inherited_facet(slot, "type")
        if typ is not MISSING:
            ok = (isinstance(value, Frame) and value.is_a(typ)) if isinstance(typ, Frame) \
                else isinstance(value, typ)
            if not ok:
                raise TypeError(f"{self.name}.{slot}: wartość {value!r} nie jest typu {typ}")
        card = self.inherited_facet(slot, "card")
        card = 1 if card is MISSING else card
        old = self.own(slot, "value")
        if card == 1:
            if self.check_changes and old is not MISSING and old == value:
                return False
            new = value
        else:
            current = [] if old is MISSING else list(old)
            if self.check_changes and value in current:
                return False
            if card is not None and len(current) >= card:
                raise ValueError(f"{self.name}.{slot}: przekroczona liczność {card}")
            new = current + [value]
        self.slots.setdefault(slot, {})["value"] = new
        demon = self.inherited_facet(slot, "if_added")
        if demon is not MISSING:
            Frame.demon_calls += 1
            try:
                demon(self, value)
            except Exception:
                # demon odrzucił wartość: przywracamy stan sprzed put
                if old is MISSING:
                    del self.slots[slot]["value"]
                else:
                    self.slots[slot]["value"] = old
                raise
        return True

In [ ]:
# Baza wiedzy: ramy klas i instancji w dziedzinie uczelni
kurs = Frame("Kurs")
kurs.define("nazwa", type=str)
kurs.define("ects", type=int, default=5)
kurs.define("wymaga", card=None)   # prerekwizyty (lista ram kursów)
kurs.define("wymaga", type=kurs)

def make_course(name, ects, requires=()):
    k = Frame(name, [kurs], instance=True)
    k.put("ects", ects)
    for r in requires:
        k.put("wymaga", r)
    return k

wdp = make_course("WstepDoProgramowania", 6)
logika = make_course("Logika", 4)
asd = make_course("AlgorytmyIStruktury", 6, [wdp])
si = make_course("SztucznaInteligencja", 5, [asd, logika])
mrw = make_course("MetodyReprezentacjiWiedzy", 5, [si])
bazy = make_course("BazyDanych", 5)

osoba = Frame("Osoba")
osoba.define("imie", type=str)
osoba.define("rok_urodzenia", type=int)
osoba.define("wiek", if_needed=lambda f: ROK_BIEZACY - f.get("rok_urodzenia")
             if f.get("rok_urodzenia") is not None else None)
osoba.define("limit_ksiazek", default=3)
osoba.define("parking", default=False)

student = Frame("Student", [osoba])
student.define("tryb", default="stacjonarny")
student.define("zajecia", default="od poniedziałku do piątku")
student.define("limit_ksiazek", default=5)
student.define("rok_studiow", type=int, default=1)
student.define("kierunek", type=str, card=2)        # najwyżej dwa kierunki
student.define("zaliczyl", type=kurs, card=None)

student_niest = Frame("StudentNiestacjonarny", [student])
student_niest.define("tryb", value="niestacjonarny")
student_niest.define("zajecia", default="zjazdy w soboty i niedziele")

pracownik = Frame("Pracownik", [osoba])
pracownik.define("limit_ksiazek", default=15)
pracownik.define("parking", default=True)

wykladowca = Frame("Wykladowca", [pracownik])
wykladowca.define("prowadzi", type=kurs, card=None)

student_pracownik = Frame("StudentPracownik", [student, pracownik])

ania = Frame("ania", [student], instance=True)
ania.put("imie", "Anna"); ania.put("rok_urodzenia", 2003); ania.put("rok_studiow", 2)
ania.put("kierunek", "informatyka")
ania.put("zaliczyl", wdp); ania.put("zaliczyl", logika); ania.put("zaliczyl", asd)

piotr = Frame("piotr", [student_niest], instance=True)
piotr.put("imie", "Piotr"); piotr.put("rok_urodzenia", 1995)
piotr.put("zaliczyl", wdp)

marek = Frame("marek", [student_pracownik], instance=True)
marek.put("imie", "Marek"); marek.put("rok_urodzenia", 2000)

nowak = Frame("nowak", [wykladowca], instance=True)
nowak.put("imie", "Ewa"); nowak.put("rok_urodzenia", 1978)
nowak.put("prowadzi", si); nowak.put("prowadzi", mrw)

print("MRO marka:", marek.linearize("mro"))
print("DFS marka:", marek.linearize("dfs"))

In [ ]:
# Dla instancji ania, piotr, marek, nowak i slotów imie, wiek, tryb, zajecia, rok_studiow,
# limit_ksiazek, parking zbudujcie tabelę: wartość, rama źródłowa i faseta (wynik lookup)

In [ ]:
# Sprawdźcie fasety type i card: spróbujcie wpisać rok urodzenia jako tekst, kurs jako napis,
# osobę jako zaliczony kurs, drugi i trzeci kierunek studiów, a także utworzyć ramę,
# której rodzicem jest instancja. Wypiszcie, co zostało przyjęte, a co odrzucone.

### Wnioski do zadania 1

(uzupełnij)

## Zadanie 2: Wartości domyślne, wyjątki i wielokrotne dziedziczenie

In [ ]:
# Wpiszcie Ani wyjątek: zajęcia według indywidualnego harmonogramu.
# Porównajcie lookup("zajecia") przed i po oraz z wynikiem dla Piotra.

In [ ]:
# Klasyczne przykłady: ptaki i diament Nixona
ptak = Frame("Ptak").define("lata", default=True).define("ma_piora", value=True)
pingwin = Frame("Pingwin", [ptak]).define("lata", default=False)
strus = Frame("Strus", [ptak]).define("lata", default=False)
tux = Frame("tux", [pingwin], instance=True)

kwakier = Frame("Kwakier").define("pacyfista", default=True)
republikanin = Frame("Republikanin").define("pacyfista", default=False)
nixon = Frame("nixon", [kwakier, republikanin], instance=True)
nixon_odwr = Frame("nixon_odwr", [republikanin, kwakier], instance=True)

class Conflict(Exception):
    pass

In [ ]:
def candidates(frame, slot):
    """Wszystkie ramy z łańcucha (kolejność MRO), które same coś mówią o slocie:
    lista par (rama, wartość). Użyjcie frame.local_answer."""
    # UZUPEŁNIJ
    raise NotImplementedError


def lookup_skeptical(frame, slot):
    """Odrzućcie kandydatów przesłoniętych przez bardziej specyficznych (rama B odpada,
    gdy inny kandydat A spełnia A.is_a(B)). Jeśli pozostali się zgadzają, zwróćcie wartość,
    w przeciwnym razie zgłoście Conflict."""
    # UZUPEŁNIJ
    raise NotImplementedError


# Zbudujcie tabelę: wiersze (tux, lata), (marek, limit_ksiazek), (marek, parking),
# (nixon, pacyfista), (nixon_odwr, pacyfista); kolumny: strategie dfs, mro, sceptyczna

In [ ]:
# Pokażcie niemonotoniczność: utwórzcie instancję ptaka, odczytajcie "lata",
# a następnie przepnijcie ją pod ramę Pingwin i odczytajcie ponownie

In [ ]:
# Ta sama wiedza o ptakach w OWL (Turtle)
PTAKI_TTL = """
@prefix : <http://example.org/ptaki#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:lata a owl:DatatypeProperty , owl:FunctionalProperty .
:Ptak a owl:Class ;
    rdfs:subClassOf [ a owl:Restriction ; owl:onProperty :lata ; owl:hasValue true ] .
:Pingwin a owl:Class ; rdfs:subClassOf :Ptak .
:kos a :Ptak .
"""
WYJATEK_TTL = """
@prefix : <http://example.org/ptaki#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
:Pingwin rdfs:subClassOf [ a owl:Restriction ; owl:onProperty :lata ; owl:hasValue false ] .
:kos a :Pingwin .
"""
P = Namespace("http://example.org/ptaki#")

In [ ]:
# Wczytajcie PTAKI_TTL, policzcie domknięcie owlrl (OWLRL_Semantics) i sprawdźcie zapytaniem
# SPARQL, jakie wartości ma kos dla :lata. Powtórzcie po dodaniu WYJATEK_TTL.
# Napiszcie zapytanie wykrywające dwie różne wartości właściwości funkcyjnej.

### Wnioski do zadania 2

(uzupełnij)

## Zadanie 3: Demony

In [ ]:
# Dopiszcie do ramy Student demona if_needed dla slotu "ects" (suma ECTS zaliczonych kursów).
# Wypiszcie wiek i ECTS dla ani, piotra i marka. Zmieńcie Ani rok urodzenia i odczytajcie wiek ponownie.

In [ ]:
# Dopiszcie demona if_added dla slotu "zaliczyl", który odrzuca kurs, jeśli student nie zaliczył
# wszystkich kursów wymaganych (slot "wymaga" kursu). Sprawdźcie: piotr+SI, piotr+ASD,
# ania+SI, ania+MRW, marek+BazyDanych. Po każdej próbie wypiszcie wynik i ECTS.

In [ ]:
# Zdefiniujcie ramę PracaDyplomowa ze slotem promotor (typ: pracownik) i dopiszcie do ramy Pracownik
# slot prace (lista prac). Dodajcie dwa demony if_added, które utrzymują relację odwrotną.
# Uruchomcie przypisanie promotora z Frame.check_changes = False, a potem True.
# Policzcie wywołania demonów (Frame.demon_calls) w obu przypadkach.

### Wnioski do zadania 3

(uzupełnij)

## Zadanie 4: Skrypty Schanka

In [ ]:
# Skrypty: role (zmienne ?X), rekwizyty, warunki wejścia, wyniki, sceny z zdarzeniami (wykonawca, czynność, obiekt)
EGZAMIN = {
    "nazwa": "egzamin ustny",
    "role": {"?S": "student", "?E": "egzaminator"},
    "rekwizyty": ["sala", "legitymacja", "zestaw pytań", "protokół"],
    "warunki_wejscia": ["?S ma wyznaczony termin", "?S przygotował się do egzaminu"],
    "wyniki": ["?S ma ocenę w protokole"],
    "sceny": [
        ("wejście", [("?S", "wchodzi", "sala"), ("?S", "okazuje", "legitymacja")]),
        ("losowanie", [("?S", "losuje", "zestaw pytań"), ("?S", "przygotowuje", "odpowiedź")]),
        ("odpowiedź", [("?S", "odpowiada", "?E"), ("?E", "zadaje", "pytanie dodatkowe"),
                       ("?E", "wystawia", "ocena")]),
        ("wyjście", [("?E", "wpisuje", "protokół"), ("?S", "wychodzi", "sala")]),
    ],
    "opcjonalne": [("?E", "zadaje", "pytanie dodatkowe")],
}

DZIEKANAT = {
    "nazwa": "zaświadczenie w dziekanacie",
    "role": {"?S": "student", "?P": "pracownik dziekanatu"},
    "rekwizyty": ["numerek", "wniosek", "zaświadczenie", "pieczątka"],
    "warunki_wejscia": ["?S potrzebuje zaświadczenia", "dziekanat jest otwarty"],
    "wyniki": ["?S ma zaświadczenie"],
    "sceny": [
        ("wejście", [("?S", "wchodzi", "dziekanat"), ("?S", "pobiera", "numerek"),
                     ("?S", "czeka", "kolejka")]),
        ("obsługa", [("?S", "składa", "wniosek"), ("?P", "drukuje", "zaświadczenie"),
                     ("?P", "podbija", "pieczątka")]),
        ("wyjście", [("?S", "odbiera", "zaświadczenie"), ("?S", "wychodzi", "dziekanat")]),
    ],
    "opcjonalne": [("?S", "czeka", "kolejka")],
}

RESTAURACJA = {
    "nazwa": "restauracja",
    "role": {"?K": "klient", "?W": "kelner"},
    "rekwizyty": ["stolik", "menu", "danie", "rachunek"],
    "warunki_wejscia": ["?K jest głodny", "?K ma pieniądze"],
    "wyniki": ["?K jest syty", "?K ma mniej pieniędzy"],
    "sceny": [
        ("wejście", [("?K", "wchodzi", "restauracja"), ("?K", "siada", "stolik")]),
        ("zamówienie", [("?W", "podaje", "menu"), ("?K", "zamawia", "?D")]),
        ("jedzenie", [("?W", "przynosi", "?D"), ("?K", "je", "?D")]),
        ("wyjście", [("?K", "prosi", "rachunek"), ("?K", "płaci", "?W"),
                     ("?K", "wychodzi", "restauracja")]),
    ],
    "opcjonalne": [("?W", "podaje", "menu")],
}
SCRIPTS = [EGZAMIN, DZIEKANAT, RESTAURACJA]

STORIES = {
    "Ania na egzaminie": [("Ania", "wchodzi", "sala"), ("Ania", "losuje", "zestaw pytań"),
                          ("Ania", "odpowiada", "dr Nowak"), ("Ania", "wychodzi", "sala")],
    "Kasia w dziekanacie": [("Kasia", "pobiera", "numerek"), ("Kasia", "składa", "wniosek"),
                            ("pani Ewa", "drukuje", "zaświadczenie"), ("Kasia", "wychodzi", "dziekanat")],
    "Jan na obiedzie": [("Jan", "wchodzi", "restauracja"), ("Jan", "zamawia", "pierogi"),
                        ("Jan", "je", "pierogi"), ("Jan", "wychodzi", "restauracja")],
    "Ola i ściąga": [("Ola", "wchodzi", "sala"), ("Ola", "losuje", "zestaw pytań"),
                     ("Ola", "ściąga", "notatki"), ("dr Nowak", "przerywa", "egzamin"),
                     ("Ola", "wychodzi", "sala")],
    "Tomek bez kolejki": [("Tomek", "wchodzi", "dziekanat"), ("Tomek", "odbiera", "zaświadczenie"),
                          ("Tomek", "składa", "wniosek"), ("Tomek", "wychodzi", "dziekanat")],
}


def unify(pattern, event, bindings):
    """Dopasowanie wzorca zdarzenia do zdarzenia z historii. Różne role muszą mieć różnych wykonawców."""
    b = dict(bindings)
    for p, e in zip(pattern, event):
        if p.startswith("?"):
            if p in b:
                if b[p] != e:
                    return None
            elif e in b.values():
                return None
            else:
                b[p] = e
        elif p != e:
            return None
    return b


def instantiate(pattern, bindings):
    return tuple(bindings.get(x, x) for x in pattern)


def flat_events(script):
    return [(scene, ev) for scene, evs in script["sceny"] for ev in evs]

In [ ]:
def match_story(script, story):
    """Dopasowuje historię do skryptu.
    Przechodźcie po zdarzeniach historii, szukając dla każdego pierwszego pasującego zdarzenia skryptu
    za bieżącą pozycją (unify z dotychczasowymi wiązaniami ról).
    Zdarzenia skryptu przeskoczone po drodze oznaczcie jako "domyślne" (albo "pominięte (opcjonalne)",
    albo "niepewne (po odstępstwie)", jeśli wcześniej wystąpiło odstępstwo).
    Zdarzenie historii bez dopasowania to odstępstwo; rozróżnijcie "zła kolejność" i "brak w skrypcie".
    Zdarzenia za ostatnim dopasowanym oznaczcie jako "nieznane".
    Zwróćcie (bindings, script_rows, story_rows)."""
    # UZUPEŁNIJ
    raise NotImplementedError

# Pokażcie wynik dla historii "Ania na egzaminie", "Ola i ściąga" (EGZAMIN)
# i "Tomek bez kolejki" (DZIEKANAT): wiązania ról, status zdarzeń historii i tabela zdarzeń skryptu

In [ ]:
def answer(script, story, question):
    """Odpowiedź na pytanie zadane jako zdarzenie (wykonawca, czynność, obiekt):
    "tak (podano wprost)", "tak (wniosek ze skryptu)" albo "nie wiadomo (...)"."""
    # UZUPEŁNIJ
    raise NotImplementedError

# Zadajcie co najmniej pytania: czy Ania okazała legitymację, czy dr Nowak wpisał ocenę do protokołu
# (u Ani i u Oli), czy zadał pytanie dodatkowe, czy Kasia odebrała zaświadczenie, czy Jan zapłacił
# (kelner nie występuje w historii, więc rola zostaje zmienną ?W), czy Jan zjadł schabowego.

In [ ]:
# Napiszcie score(script, story) (np. odsetek dopasowanych zdarzeń historii) i choose_script(story),
# która wybiera skrypt o najwyższym wyniku. Zbudujcie tabelę wyników dla wszystkich historii i skryptów.

### Wnioski do zadania 4

(uzupełnij)

## Zadanie 5: Sieć powiązań i rozprzestrzenianie aktywacji

In [ ]:
EDGES_TXT = """
lekarz;szpital;0.9
lekarz;pacjent;0.9
lekarz;pielęgniarka;0.8
lekarz;choroba;0.7
lekarz;recepta;0.7
lekarz;stetoskop;0.6
lekarz;fartuch;0.5
lekarz;chirurg;0.7
lekarz;przychodnia;0.7
lekarz;dyżur;0.4
lekarz;medycyna;0.8
szpital;pacjent;0.8
szpital;pielęgniarka;0.8
szpital;karetka;0.7
szpital;operacja;0.7
szpital;łóżko;0.5
szpital;dyżur;0.5
szpital;oddział;0.6
pielęgniarka;zastrzyk;0.6
pielęgniarka;pacjent;0.7
pielęgniarka;fartuch;0.4
pacjent;choroba;0.7
pacjent;ból;0.5
pacjent;kolejka;0.4
choroba;gorączka;0.6
choroba;lek;0.7
choroba;grypa;0.6
grypa;gorączka;0.7
grypa;zima;0.4
lek;apteka;0.8
lek;recepta;0.7
lek;tabletka;0.7
apteka;recepta;0.6
apteka;farmaceuta;0.8
chirurg;operacja;0.9
chirurg;medycyna;0.5
operacja;skalpel;0.7
karetka;syrena;0.6
karetka;samochód;0.5
karetka;ratownik;0.8
ratownik;wypadek;0.6
wypadek;samochód;0.6
wypadek;droga;0.5
przychodnia;kolejka;0.5
przychodnia;rejestracja;0.6
medycyna;uczelnia;0.4
student;uczelnia;0.9
student;egzamin;0.8
student;wykład;0.7
student;sesja;0.7
student;akademik;0.5
student;legitymacja;0.5
student;stypendium;0.5
wykładowca;wykład;0.8
wykładowca;uczelnia;0.7
wykładowca;egzamin;0.6
egzamin;ocena;0.8
egzamin;sesja;0.8
egzamin;stres;0.6
ocena;protokół;0.5
uczelnia;dziekanat;0.6
uczelnia;biblioteka;0.6
dziekanat;kolejka;0.5
dziekanat;zaświadczenie;0.7
dziekanat;legitymacja;0.5
biblioteka;książka;0.9
wykład;notatki;0.6
stres;ból;0.3
akademik;stołówka;0.4
restauracja;kelner;0.9
restauracja;menu;0.8
restauracja;obiad;0.7
restauracja;kucharz;0.7
restauracja;rachunek;0.7
kelner;rachunek;0.6
kelner;napiwek;0.6
kucharz;kuchnia;0.8
kucharz;fartuch;0.5
kuchnia;nóż;0.5
nóż;skalpel;0.4
obiad;zupa;0.6
obiad;pierogi;0.5
stołówka;obiad;0.7
stołówka;restauracja;0.4
rachunek;pieniądze;0.6
napiwek;pieniądze;0.5
stypendium;pieniądze;0.5
pieniądze;bank;0.7
samochód;droga;0.7
autobus;przystanek;0.8
autobus;kierowca;0.6
kierowca;samochód;0.7
autobus;droga;0.4
fartuch;biały;0.6
biały;śnieg;0.7
śnieg;zima;0.9
zima;sesja;0.4
łóżko;sen;0.7
sen;zmęczenie;0.5
dyżur;zmęczenie;0.5
sesja;zmęczenie;0.4
"""

In [ ]:
# Zbudujcie nieskierowany graf ważony G z EDGES_TXT (format: pojęcie;pojęcie;waga).
# Wypiszcie liczbę pojęć i krawędzi, spójność, średnicę i pojęcia o najwyższym stopniu.

In [ ]:
def spread(G, sources, decay=0.4, threshold=0.1, max_steps=5, fan_out=False):
    """Rozprzestrzenianie aktywacji impulsami. Źródła mają aktywację 1.
    W każdym kroku każdy węzeł o aktywacji >= threshold, który jeszcze nie wysyłał,
    wysyła do sąsiadów act[i] * w_ij * decay (przy fan_out=True dodatkowo dzielone przez stopień i).
    Aktywację obcinamy do 1. Zwróćcie pd.Series posortowaną malejąco."""
    # UZUPEŁNIJ
    raise NotImplementedError

# Policzcie aktywację ze źródeł ["lekarz"], ["szpital"] i ["lekarz", "szpital"].
# Zbudujcie tabelę 12 najbardziej aktywnych pojęć (bez źródeł) z kolumną "torowanie":
# aktywacja przy dwóch źródłach minus większa z aktywacji przy jednym źródle.

In [ ]:
# Zbadajcie wpływ parametrów dla źródła "lekarz": decay w {0.3, 0.4, 0.6, 0.8} i próg w {0.05, 0.1, 0.2}
# bez normalizacji, a także fan_out=True przy decay=1.0 i progach {0.01, 0.02, 0.05}.
# Podajcie liczbę pojęć z aktywacją >= próg, aktywację "pielęgniarka", "kucharz" i "kelner"
# oraz trzy najbardziej aktywne pojęcia.

In [ ]:
# Narysujcie sieć (nx.spring_layout z seed=RANDOM_STATE) z kolorem węzła równym aktywacji,
# obok siebie dla źródła "lekarz" i dla źródeł "lekarz" i "szpital"

In [ ]:
# Pary pojęć z tłumaczeniem na synsety angielskiego WordNetu (znaczenia wybrane ręcznie)
PAIRS = [
    ("lekarz", "pielęgniarka", "doctor.n.01", "nurse.n.01"),
    ("lekarz", "chirurg", "doctor.n.01", "surgeon.n.01"),
    ("lekarz", "pacjent", "doctor.n.01", "patient.n.01"),
    ("lekarz", "szpital", "doctor.n.01", "hospital.n.01"),
    ("lekarz", "kelner", "doctor.n.01", "waiter.n.01"),
    ("lekarz", "kucharz", "doctor.n.01", "cook.n.01"),
    ("szpital", "przychodnia", "hospital.n.01", "clinic.n.03"),
    ("szpital", "restauracja", "hospital.n.01", "restaurant.n.01"),
    ("lek", "apteka", "medicine.n.02", "drugstore.n.01"),
    ("grypa", "gorączka", "influenza.n.01", "fever.n.01"),
    ("karetka", "samochód", "ambulance.n.01", "car.n.01"),
    ("skalpel", "nóż", "scalpel.n.01", "knife.n.01"),
    ("student", "wykładowca", "student.n.01", "lecturer.n.02"),
    ("kelner", "rachunek", "waiter.n.01", "bill.n.02"),
    ("śnieg", "zima", "snow.n.02", "winter.n.01"),
    ("biblioteka", "książka", "library.n.01", "book.n.01"),
]

In [ ]:
# Dla każdej pary z PAIRS policzcie: średnią aktywację (b ze źródła a oraz a ze źródła b),
# path_similarity i wup_similarity synsetów oraz najniższy wspólny hiperonim.
# Policzcie korelację Spearmana aktywacji z każdą z miar WordNetu.

### Wnioski do zadania 5

(uzupełnij)

## Zadanie 6: Pytania kontrolne

Odpowiedzcie na pytania kontrolne z instrukcji, po jednym do trzech zdań na pytanie.

(uzupełnij)